# 129 — MCP: tools, resources y prompts

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (a) **tool** — tiene efectos y la invoca el modelo cuando la tarea lo
pide. (b) **resource** — estado de solo lectura direccionable; la aplicación decide
inyectarlo. (c) **prompt** — flujo empaquetado que lanza el usuario. (d) **tool** con
efecto destructivo → el host debería exigir consentimiento explícito. (e)
**resource** (URI parametrizada, p. ej. `incident://4812/history`).

**Ejercicio 2.** Lo importante: la description explica *cuándo* usarla (el modelo
elige tools leyendo descripciones) y el enum cierra la severidad — un string libre
invita a "urgentísima".

**Ejercicio 3.** Las dos capas quedan separadas: `-32601` (method/tool not found) es
para el *cliente*; `isError: true` viaja como contenido para que el *modelo* lea el
motivo y corrija los argumentos.

**Ejercicio 4.** `received` = llega la petición y el modelo propone un `tools/call`;
`validated` = argumentos contra inputSchema; `waiting_approval → completed` es el
**consentimiento humano** que la spec exige para tools sensibles: la transición
`waiting_approval → completed` (con `approved: true` registrado) es el análogo exacto.


In [ ]:
result = run_lab("workflow", seed=129)
assert result["kind"] == "workflow"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 2
tool_def = {
    "name": "crear_incidencia",
    "description": ("Crea una incidencia nueva en el gestor. Úsala cuando el usuario "
                    "reporte un problema que no exista ya en el tablero."),
    "inputSchema": {
        "type": "object",
        "properties": {
            "titulo": {"type": "string", "minLength": 1},
            "severidad": {"type": "string",
                           "enum": ["baja", "media", "alta", "critica"]},
            "etiquetas": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["titulo", "severidad"],
    },
}

# Ejercicio 3
def _crear_incidencia(args):
    sev = args.get("severidad")
    if sev not in tool_def["inputSchema"]["properties"]["severidad"]["enum"]:
        return {"content": [{"type": "text",
                              "text": f"severidad inválida: {sev!r}"}],
                "isError": True}
    return {"content": [{"type": "text",
                          "text": f"INC-001 creada: {args['titulo']} [{sev}]"}],
            "isError": False}

TOOLS = {"crear_incidencia": (tool_def, _crear_incidencia)}

def handle(request):
    if request["method"] == "tools/list":
        return {"result": {"tools": [d for d, _ in TOOLS.values()]}}
    if request["method"] == "tools/call":
        name = request["params"]["name"]
        if name not in TOOLS:
            return {"error": {"code": -32601, "message": f"tool not found: {name}"}}
        return {"result": TOOLS[name][1](request["params"].get("arguments", {}))}
    return {"error": {"code": -32601, "message": "method not found"}}

print(handle({"method": "tools/list"})["result"]["tools"][0]["name"])
print(handle({"method": "tools/call", "params": {"name": "crear_incidencia",
      "arguments": {"titulo": "502 en pagos", "severidad": "alta"}}}))
print(handle({"method": "tools/call", "params": {"name": "crear_incidencia",
      "arguments": {"titulo": "x", "severidad": "urgentísima"}}}))

# Ejercicio 4
result = run_lab("workflow", seed=129)
eventos = result["result"]["events"]
assert result["result"]["approved"] is True
print("transiciones:", [(e["from"], e["to"]) for e in eventos])
print("consentimiento ≙ waiting_approval → completed")


## Reflexión

1. El laboratorio registra una aprobación antes de completar el workflow. ¿A cuál requisito de la spec de MCP (consentimiento en el host para tools sensibles) corresponde ese paso, y qué tool del ejemplo lo necesitaría?
2. ¿Por qué "archivo no encontrado" debe viajar como `isError: true` dentro de un result y no como error JSON-RPC? ¿Qué puede hacer el modelo en el primer caso que no puede en el segundo?
3. Si conviertes un resource (`file:///repo/README.md`) en una tool `read_readme()`, ¿qué decisión transferiste de la aplicación al modelo y qué riesgo nuevo aparece?
